<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_1_Trees_Data_and_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Trees: The Foundation of Ensembles

**Attribution**: Exercises adapted from *Introduction to Modern Statistics (2e)* Chapter 9.6.
Source: [OpenIntro IMS](https://openintro-ims.netlify.app/model-logistic)

**License**: This work is a derivative of [Introduction to Modern Statistics (2e)](https://openintro-ims.netlify.app/) by Mine Çetinkaya-Rundel and Johanna Hardin, used under a [Creative Commons Attribution-ShareAlike 3.0 Unported (CC BY-SA 3.0 US)](https://creativecommons.org/licenses/by-sa/3.0/us/) license.

---

## Introduction
In this notebook, we move from the probability-based logic of Logistic Regression to the "flowchart" logic of **Decision Trees**. Decision trees are highly interpretable, making them the most easily explainable machine learning models for non-technical audiences.

However, we will also see their primary weakness: **overfitting**. This notebook will serve as our "on-ramp" for building high-performance ensembles like Random Forests and Boosting models later in this series.

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Load and Explore** the Wisconsin Breast Cancer (Diagnostic) dataset.
2. **Implement** a Decision Tree classifier using `scikit-learn`.
3. **Visualize** the model's decision-making process with a tree diagram.
4. **Evaluate** the impact of tree depth on model accuracy and overfitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set style
sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Preparation: Wisconsin Breast Cancer Dataset

We are working with measurements taken from digitized images of a fine needle aspirate (FNA) of a breast mass. These describe characteristics of the cell nuclei present in the image.

**Goal**: Predict whether a mass is **Malignant** (1) or **Benign** (0).

In [ ]:
# Load the dataset
data = load_breast_cancer(as_frame=True)
df = data.frame

print(f"Dataset Shape: {df.shape}")
print(f"Target Distribution:\n{df['target'].value_counts(normalize=True)}")
df.head()

### Exploratory Data Analysis (EDA)
Decision trees find the best feature and "split point" to divide the data. Let's look at one feature, `mean concave points`, and see how it differs between Malignant and Benign tumors.

In [ ]:
sns.violinplot(x='target', y='mean concave points', data=df)
plt.title("Distribution of Mean Concave Points by Malignancy")
plt.xticks([0, 1], ['Benign', 'Malignant'])
plt.show()

### Train/Test Split
To evaluate how our model will perform on brand-new data, we must split our data into training and testing sets.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

---
## 2. The Building Block: A Single Decision Tree

A decision tree finds a specific feature and threshold to split the data. For example: "If `mean concave points` > 0.05, go left; otherwise, go right."

### Growing the Tree
We use `DecisionTreeClassifier`. We will start by limiting the depth of the tree to `max_depth=3` to keep it interpretable.

In [ ]:
# Initialize and fit the model
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

### Visualizing the Logic
The most powerful feature of a decision tree is that we can literally look at its decision-making process.

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(clf, 
          feature_names=data.feature_names, 
          class_names=['Benign', 'Malignant'], 
          filled=True, 
          rounded=True)
plt.title("Decision Tree Flowchart (Max Depth = 3)")
plt.show()

### How to Read the Tree:
1. **Gini**: A measure of "purity." A Gini of 0.0 means every mass in that node is the same class (perfectly pure).
2. **Samples**: How many mass mass masses are in that node.
3. **Value**: The count of [Benign, Malignant] samples in that node.
4. **Color**: The darker the color, the more "pure" the node is for that class.

---
## 3. The Overfitting Problem

If we don't limit the depth, the tree will keep splitting until every single node is pure. While this looks great on the training data, it often fails on the testing data because the model has "memorized" the noise in the training set.

**Student Task**: Compare the accuracy of a "Limited" tree vs. an "Unconstrained" tree.

In [ ]:
# Full (Overfitted) Tree
full_clf = DecisionTreeClassifier(random_state=42)
full_clf.fit(X_train, y_train)

# Predictions
limited_train_acc = accuracy_score(y_train, clf.predict(X_train))
limited_test_acc = accuracy_score(y_test, clf.predict(X_test))

full_train_acc = accuracy_score(y_train, full_clf.predict(X_train))
full_test_acc = accuracy_score(y_test, full_clf.predict(X_test))

print(f"Limited Tree (Depth 3): Training Acc = {limited_train_acc:.3f}, Test Acc = {limited_test_acc:.3f}")
print(f"Full Tree (No Limit): Training Acc = {full_train_acc:.3f}, Test Acc = {full_test_acc:.3f}")

### Discussion: The Complexity Tradeoff
Notice that the "Full Tree" might reach **100% accuracy** on the training data. However, the testing accuracy is often lower than the limited tree. This gap between training and testing performance is the hallmark of **Overfitting**.

In the next notebook, we will learn how to use **Bagging** and **Random Forests** to harness the power of deep trees without suffering from this instability.